In [0]:
import logging
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DoubleType,DateType

In [0]:
class BronzeIngestion:

    def __init__(self, spark, data_path, catalog_name):
        self.spark = spark
        self.data_path = data_path
        self.catalog_name = catalog_name

        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter(
                "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
    
    def define_schema(self):

        self.logger.info("Defining Bronze schema")

        bronze_schema = StructType([
            StructField("transaction_id", StringType(), True),
            StructField("transaction_date", DateType(), True),
            StructField("transaction_year", IntegerType(), True),
            StructField("transaction_quarter", IntegerType(), True),
            StructField("transaction_month", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("product_category", StringType(), True),
            StructField("quantity_unit", StringType(), True),
            StructField("supplier_name", StringType(), True),
            StructField("supplier_country", StringType(), True),
            StructField("supplier_reliability_score", DoubleType(), True),
            StructField("refinery_name", StringType(), True),
            StructField("destination_city", StringType(), True),
            StructField("transportation_mode", StringType(), True),
            StructField("ordered_quantity", DoubleType(), True),
            StructField("demand_quantity", DoubleType(), True),
            StructField("available_inventory", DoubleType(), True),
            StructField("unit_price_usd", DoubleType(), True),
            StructField("product_cost_usd", DoubleType(), True),
            StructField("transportation_cost_usd", DoubleType(), True),
            StructField("total_cost_usd", DoubleType(), True),
            StructField("expected_lead_time_days", IntegerType(), True),
            StructField("actual_lead_time_days", IntegerType(), True),
            StructField("delay_days", IntegerType(), True),
            StructField("is_delayed", IntegerType(), True),
            StructField("is_stockout", IntegerType(), True),
            StructField("quality_status", StringType(), True),
            StructField("quality_score", DoubleType(), True),
            StructField("disruption_type", StringType(), True),
            StructField("delivery_status", StringType(), True),
            StructField("ingestion_timestamp", StringType(), True),
            StructField("source_system", StringType(), True),
            StructField("operation_type", StringType(), True),
        ])

        self.logger.info("Bronze schema defined successfully")

        return bronze_schema
    
    def create_dataframe(self):

        self.logger.info(f"Reading source file: {self.data_path}")
        try:
            bronze_schema = self.define_schema()
            bronze_df = self.spark.read.csv(
                path=self.data_path,
                schema=bronze_schema,
                header=True,
                sep=","
            )

            bronze_df = bronze_df.withColumn("ingested_at",F.current_timestamp())

            self.logger.info("DataFrame created successfully")

            return bronze_df

        except Exception as e:

            raise self.logger.error(
                f"Failed to create DataFrame: {str(e)}"
            )
    
    def create_table(self):

        table_name = f"{self.catalog_name}.bronze.bronze_table"
        self.logger.info(f"Starting Bronze table creation: {table_name}")
        try:
            bronze_df = self.create_dataframe()
            record_count = bronze_df.count()
            self.logger.info(f"Records read from source: {record_count}")
            (
                bronze_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(table_name)
            )

            self.logger.info(f"Bronze table created successfully: {table_name}")
            return True
        except Exception as e:
            raise self.logger.error(
                f"Bronze table creation failed: {str(e)}"
            )
    
    def run(self):
        self.logger.info("========== Bronze ingestion started ==========")
        try:
            result = self.create_table()
            if result:
                self.logger.info("========== Bronze ingestion completed successfully ===========")
                return result
        except Exception as e:
                raise self.logger.error("========== Bronze ingestion failed ==========")

In [0]:
DATA_PATH = "/Volumes/oag/bronze/raw/supply_chain_dataset.csv"
CATALOG_NAME = "oag"

bronze_ingestion = BronzeIngestion(
    spark=spark,
    data_path=DATA_PATH,
    catalog_name=CATALOG_NAME
)
bronze_ingestion.run()